# NH3 / H2 Qubit Hamiltonian (Colab Minimal)

This notebook now defers all logic to the repository code. Workflow:

1. Install pinned dependencies (Qiskit 2.x, qiskit-nature, PySCF).
2. Clone the repo.
3. Run the CLI script for NH3 (active space, 6 qubits) or H2 (4 qubits).
4. (Optional) Use fallback only (`--force-precomputed`) if PySCF fails.

See repository README for details and provenance notes.


In [ ]:
# Install pinned dependencies (single step) and import modules
import sys, subprocess, importlib
pkgs = ['qiskit==2.1.2','qiskit-nature==0.7.2','pyscf==2.6.1']
subprocess.check_call([sys.executable,'-m','pip','install','--upgrade','--no-cache-dir']+pkgs)

# Core imports (moved here as requested)
from qiskit_nature.units import DistanceUnit
from qiskit_nature.second_q.drivers import PySCFDriver
from qiskit_nature.second_q.problems import ElectronicStructureProblem
from qiskit_nature.second_q.transformers import ActiveSpaceTransformer
from qiskit_nature.second_q.mappers import JordanWignerMapper
from qiskit.quantum_info import SparsePauliOp

print('\nVersions:')
for mod in ['qiskit','qiskit_nature','pyscf']:
    try:
        m = importlib.import_module(mod)
        print(f'  {mod}:', getattr(m,'__version__','?'))
    except Exception as e:
        print(f'  {mod}: MISSING ({e})')

Direct NH3 6-qubit active-space build and Pauli expansion (no error handling).

In [ ]:
# NH3 Pauli expansion (direct, minimal) with optional padding
import os, sys, itertools
from qiskit_nature.units import DistanceUnit
from qiskit_nature.second_q.drivers import PySCFDriver
from qiskit_nature.second_q.problems import ElectronicStructureProblem
from qiskit_nature.second_q.transformers import ActiveSpaceTransformer
from qiskit_nature.second_q.mappers import JordanWignerMapper

DEFAULT_MIN = 400
try:
    user_in = input(f"Minimum number of Pauli strings to print (ENTER for {DEFAULT_MIN}): ").strip()
    DESIRED_MIN = int(user_in) if user_in else DEFAULT_MIN
    if DESIRED_MIN < 1: DESIRED_MIN = DEFAULT_MIN
except Exception:
    DESIRED_MIN = DEFAULT_MIN
print(f"Target minimum terms: {DESIRED_MIN}")

# Build NH3 active-space (6 qubits)
geom = (
    'N  0.0000  0.0000  0.0000;'
    ' H  0.9377  0.0000 -0.3816;'
    ' H -0.4688  0.8119 -0.3816;'
    ' H -0.4688 -0.8119 -0.3816'
)
driver = PySCFDriver(atom=geom, basis='sto3g', charge=0, spin=0, unit=DistanceUnit.ANGSTROM)
res = driver.run()
problem = res if isinstance(res, ElectronicStructureProblem) else ElectronicStructureProblem(res)
transformer = ActiveSpaceTransformer(num_electrons=4, num_spatial_orbitals=3)
problem = transformer.transform(problem)
ferm = problem.hamiltonian.second_q_op()
mapper = JordanWignerMapper()
op = mapper.map(ferm)

labels = op.paulis.to_labels()
coeffs = op.coeffs
terms = []
seen = set()
for lbl, c in zip(labels, coeffs):
    if set(lbl) == {'I'}: continue
    seen.add(lbl)
    if abs(c.imag) > 1e-12:
        terms.append((lbl, c.real, c.imag))
    else:
        terms.append((lbl, c.real, 0.0))

# Pad with zero-real labels if needed
if len(terms) < DESIRED_MIN:
    for cand in (''.join(p) for p in itertools.product('IXYZ', repeat=op.num_qubits)):
        if cand in seen or set(cand)=={'I'}: continue
        seen.add(cand)
        terms.append((cand, 0.0, 0.0))
        if len(terms) >= DESIRED_MIN: break

# Print (non-zero first, then zero padding sorted)
nonzero = [t for t in terms if abs(t[1])>1e-12 or abs(t[2])>1e-12]
zeroed = [t for t in terms if abs(t[1])<=1e-12 and abs(t[2])<=1e-12]
zeroed.sort(key=lambda x: x[0])
final = nonzero + zeroed
for lbl,r,i in final:
    if abs(i)>1e-12:
        print(f'({r:+.12f}{i:+.12f}j) * {lbl}')
    else:
        print(f'{r:+.12f} * {lbl}')
print(f'\nTotal printed: {len(final)} (physical non-zero: {len(nonzero)})')

## Note
Padding adds zero-coefficient Pauli strings to reach the requested minimum; physics unaffected.